## Cell 1: Setup

In [ ]:
!nvidia-smi -L
!df -h /content/drive 2>/dev/null | tail -1   # Google Drive space: need about 12 GB free in total
from google.colab import drive, files
drive.mount('/content/drive')
MYDRIVE, CONTENT, UPLOAD = '/content/drive/MyDrive', '/content', files.upload

# Finds the code in Drive (any name like "gan_vs_diffusion (1).zip", in a sub-folder,
# or already unzipped); if it is not in Drive, asks you to upload the zip.
import os, glob, shutil, subprocess, zipfile
WORK = f'{MYDRIVE}/gan_vs_diffusion_work'
os.makedirs(WORK, exist_ok=True)
CODE = f'{CONTENT}/gan_vs_diffusion'

def install_from_zip(z):
    tmp = f'{CONTENT}/_code'
    shutil.rmtree(tmp, ignore_errors=True)
    zipfile.ZipFile(z).extractall(tmp)
    hit = glob.glob(f'{tmp}/**/ddpm/train.py', recursive=True)
    if not hit:
        raise SystemExit(f'{z} does not contain the project code (ddpm/train.py missing).')
    shutil.copytree(os.path.dirname(os.path.dirname(hit[0])), CODE, dirs_exist_ok=True)

if not os.path.exists(f'{CODE}/ddpm/train.py'):
    zips = [p for d in range(4) for p in glob.glob(MYDRIVE + '/*' * d + '/gan_vs_diffusion*.zip')]
    dirs = [os.path.dirname(os.path.dirname(p)) for d in range(4)
            for p in glob.glob(MYDRIVE + '/*' * d + '/ddpm/train.py')]
    if zips:
        print('Found zip in Drive:', zips[0]); install_from_zip(zips[0])
    elif dirs:
        print('Found code folder in Drive:', dirs[0]); shutil.copytree(dirs[0], CODE, dirs_exist_ok=True)
    else:
        print('Code not found in Google Drive. Pick gan_vs_diffusion.zip from your computer:')
        up = UPLOAD()
        name = next(n for n in up if n.endswith('.zip'))
        with open(f'{CONTENT}/{name}', 'wb') as f:
            f.write(up[name])
        install_from_zip(f'{CONTENT}/{name}')
        shutil.copy(f'{CONTENT}/{name}', f'{MYDRIVE}/gan_vs_diffusion.zip')   # keep a copy in Drive for next time
os.chdir(CODE)
print('Code ready in', CODE)

!pip install -q tifffile torch-fidelity ninja click psutil imageio scikit-image
DATA = '/content/bbbc_128'   # fast local disk (writing thousands of images to Drive is very slow)

## Cell 2: Data download 

In [ ]:
import os
if os.path.exists(f'{WORK}/bbbc_128.zip'):
    print('Prepared data already saved in Drive - no download needed, go to Cell 3.')
else:
    !mkdir -p /content/datasets && cd /content/datasets && \
     wget -q -nc https://data.broadinstitute.org/bbbc/BBBC038/stage1_train.zip && \
     wget -q -nc https://data.broadinstitute.org/bbbc/BBBC039/images.zip && \
     wget -q -nc https://data.broadinstitute.org/bbbc/BBBC039/masks.zip && \
     wget -q -nc https://data.broadinstitute.org/bbbc/BBBC039/metadata.zip && \
     mkdir -p stage1_train bbbc039 && unzip -oq stage1_train.zip -d stage1_train && \
     unzip -oq images.zip -d bbbc039 && unzip -oq masks.zip -d bbbc039 && unzip -oq metadata.zip -d bbbc039
    !ls /content/datasets/stage1_train | head -3; ls /content/datasets/bbbc039

## Cell 3: Unpacking prepared data from Drive

In [ ]:
import os
if os.path.exists(f'{WORK}/bbbc_128.zip'):
    !unzip -oq {WORK}/bbbc_128.zip -d /content
    print('Restored prepared data from Drive.')
else:
    !python data/prepare_bbbc.py --bbbc038 /content/datasets/stage1_train --bbbc039 /content/datasets/bbbc039 \
        --out {DATA} --crop 128 --sizes 5 25 100 --seeds 0 --heldout-frac 0
    # save everything as ONE file in Drive (fast), so restarts only need to unzip it
    if os.path.exists(f'{DATA}/manifest.json'):
        !cd /content && zip -qr -0 bbbc_128.zip bbbc_128 && cp bbbc_128.zip {WORK}/bbbc_128.zip
        print('Prepared data saved to Drive as bbbc_128.zip')
    else:
        print('Data preparation did not finish - send a screenshot of the error above.')
!ls {DATA} {DATA}/subsets/seed0

## Cell 4: Train


In [ ]:
for n in [5, 25, 100]:
    !python ddpm/train.py --data-root {DATA} --n {n} --seed 0 --steps 10000 --batch-gpu 32 \
        --ckpt-every 2500 --sample-every 2500 --outdir {WORK}/runs/ddpm/n{n}-seed0

## Cell 5: Evaluate

In [ ]:
VAL, TEST = f'{DATA}/bbbc039/validation/images', f'{DATA}/bbbc039/test/images'
for n in [5, 25, 100]:
    !python eval/select_best.py --kind ddpm --run {WORK}/runs/ddpm/n{n}-seed0 \
        --real {VAL} --real-final {TEST} --n-screen 256 --n-final 300 --screen-steps 100 --every 2 --batch 128
    if _exit_code != 0:
        raise RuntimeError(f'Evaluation of n={n} failed - see the error above (the notebook stops here).')

## Results

In [ ]:
import json, glob
for f in sorted(glob.glob(f'{WORK}/runs/**/final_metrics.json', recursive=True)):
    m = json.load(open(f))
    print(f.split('runs/')[1].split('/final')[0].ljust(32),
          {k: round(v, 4) for k, v in m.items() if isinstance(v, float)})

## Switch the GPU off

In [ ]:
from google.colab import runtime
runtime.unassign()